# Import Libraries

In [1]:
import os

import pandas as pd
import numpy as np
import nltk
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder


In [2]:
import sys, time, math, re, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python {sys.version.split()} | PyTorch {torch.__version__} | Device: {device}")

Python ['3.10.10', '(tags/v3.10.10:aad5f6a,', 'Feb', '7', '2023,', '17:20:36)', '[MSC', 'v.1929', '64', 'bit', '(AMD64)]'] | PyTorch 2.12.0+cu126 | Device: cuda


In [3]:
RANDOM_SEED = 42

# Read Data

In [4]:
# import os
# import pandas as pd
# from sklearn.model_selection import train_test_split

# RANDOM_SEED = 42

# path = "../../data/gold_labeled_data"

# df = pd.read_excel(
#     os.path.join(path, "app_reviews_ru_gold_labeled.xlsx"),
#     sheet_name="FINAL_GOLD"
# )

# df = df.drop(
#     columns=["Unnamed: 0", "lable_silver_common_AI", "summary_silver_GPT"],
#     errors="ignore"
# )

# df = df.dropna(subset=["text", "label_gold"]).reset_index(drop=True)

# train_df, test_df = train_test_split(
#     df,
#     test_size=0.15,
#     random_state=RANDOM_SEED,
#     shuffle=True,
#     stratify=df["label_gold"]
# )

# train_df = train_df.reset_index(drop=True)
# test_df = test_df.reset_index(drop=True)

# train_df.to_excel(os.path.join(path, "train.xlsx"), index=False)
# test_df.to_excel(os.path.join(path, "test.xlsx"), index=False)

# print("Train shape:", train_df.shape)
# print("Test shape:", test_df.shape)

# print("\nTrain class distribution:")
# print(train_df["label_gold"].value_counts(normalize=True).round(4))

# print("\nTest class distribution:")
# print(test_df["label_gold"].value_counts(normalize=True).round(4))

In [5]:
path = "../../data/gold_labeled_data"
train = pd.read_excel(os.path.join(path, "train.xlsx"))
test = pd.read_excel(os.path.join(path, "test.xlsx"))
print("Number of rows and columns in the train data set:", train.shape)
print("Number of rows and columns in the test data set:", test.shape)
train.head()

Number of rows and columns in the train data set: (2714, 6)
Number of rows and columns in the test data set: (480, 6)


,app_name,text,rating,thumbs_up_count,label_gold,summary_gold
0,Avito,программа Авито очень помогает в продаже б.у т...,5,0,user_experience,Положительный опыт использования приложения
1,VK,Скачала и пользовалась с 2015 года. Новый отзы...,1,24,user_experience,Недовольство модерацией и поддержкой
2,Yandex Maps,Сотрудники Яндекс Поддержки продались и сотруд...,1,5,user_experience,Мешает навязчивая реклама
3,Telegram,"прошу, сделайте так чтоб подарки можно было пр...",5,0,feature_request,Разрешить продажу подарков в любое время
4,Yandex Maps,настройте приложение опять заедает и вылетает,1,3,bug_report,Приложение вылетает


# Label encoding

In [6]:
label_dict = {
    "bug_report": 0,
    "feature_request": 1,
    "rating": 2,
    "user_experience": 3
}

In [7]:
train["label_num"] = train["label_gold"].map(label_dict).astype("int16")
test["label_num"] = test["label_gold"].map(label_dict).astype("int16")

In [8]:
train["label_num"].value_counts()

label_num
3    1289
0     705
2     575
1     145
Name: count, dtype: int64

In [9]:
from gensim.utils import simple_preprocess

train['text'].head().apply(simple_preprocess)

0    [программа, авито, очень, помогает, продаже, т...
1    [скачала, пользовалась, года, новый, отзыв, пи...
2    [сотрудники, яндекс, поддержки, продались, сот...
3    [прошу, сделайте, так, чтоб, подарки, можно, б...
4    [настройте, приложение, опять, заедает, вылетает]
Name: text, dtype: object

In [10]:
import re
import numpy as np
import pandas as pd

from razdel import tokenize
import pymorphy2

from sklearn.feature_extraction.text import TfidfVectorizer
import compress_fasttext


# -------------------------
# 1) Морфология и токенизация
# -------------------------
morph = pymorphy2.MorphAnalyzer()

# при желании можно добавить свои стоп-слова
stop_words = set()

def normalize_ru(word: str) -> str:
    """
    Лемматизация через PyMorphy2.
    """
    word = word.lower().replace("ё", "е")
    return morph.parse(word)[0].normal_form


def text_preproc_ru(text: str) -> str:
    """
    Токенизация Razdel -> очистка -> лемматизация -> строка токенов.
    """
    text = "" if text is None else str(text)
    out = []

    for t in tokenize(text):
        w = t.text.lower().replace("ё", "е")

        # оставляем только слова/смешанные токены с кириллицей или латиницей
        if not re.search(r"[а-яa-z]", w, flags=re.I):
            continue

        if w in stop_words:
            continue

        out.append(normalize_ru(w))

    return " ".join(out)



In [11]:
train['tokens'] = train['text'].apply(text_preproc_ru)
test['tokens'] = test['text'].apply(text_preproc_ru)

In [12]:
train

,app_name,text,rating,thumbs_up_count,label_gold,summary_gold,label_num,tokens
0,Avito,программа Авито очень помогает в продаже б.у т...,5,0,user_experience,Положительный опыт использования приложения,3,программа авить очень помогать в продажа б у т...
1,VK,Скачала и пользовалась с 2015 года. Новый отзы...,1,24,user_experience,Недовольство модерацией и поддержкой,3,скачать и пользоваться с год новый отзыв писат...
2,Yandex Maps,Сотрудники Яндекс Поддержки продались и сотруд...,1,5,user_experience,Мешает навязчивая реклама,3,сотрудник яндекс поддержка продаться и сотрудн...
3,Telegram,"прошу, сделайте так чтоб подарки можно было пр...",5,0,feature_request,Разрешить продажу подарков в любое время,1,просить сделать так чтоб подарок можно быть пр...
4,Yandex Maps,настройте приложение опять заедает и вылетает,1,3,bug_report,Приложение вылетает,0,настроить приложение опять заедать и вылетать
...,...,...,...,...,...,...,...,...
2709,Avito,сделка состоялась без проблем👍,5,0,user_experience,Положительный опыт использования приложения,3,сделка состояться без проблем👍
2710,Avito,"Заставили меня обновить приложение,хоть и не х...",1,1,bug_report,Приложение не открывается или не запускается,0,заставить я обновить приложение хоть и не хоте...
2711,VK,"максимально печальное приложение, хуже просто ...",1,5,bug_report,Недовольство модерацией и поддержкой,0,максимально печальный приложение плохой просто...
2712,Yandex Go,"нету другого сервиса , таксисты не приезжают н...",1,0,user_experience,Недовольство платными функциями,3,нету другой сервис таксист не приезжать на лок...


In [13]:
test

,app_name,text,rating,thumbs_up_count,label_gold,summary_gold,label_num,tokens
0,VK,"с каждым новым обновлением всё хуже, даже сооб...",1,0,user_experience,Проблемы с чатами и сообщениями,3,с каждый новый обновление всё плохой даже сооб...
1,Telegram,"приложение супер, только почему-то после того ...",5,0,user_experience,Приложение работает некорректно,3,приложение супер только почему-то после тот ка...
2,VK,при включении музыки постоянно виснит и вылета...,1,1,bug_report,Приложение лагает и тормозит,0,при включение музыка постоянно виснит и вылета...
3,WhatsApp,после обновления приложения не работает не мог...,1,0,bug_report,Приложение работает некорректно,0,после обновление приложение не работать не моч...
4,VK,очень неудобное приложение. проблем внутри куч...,1,0,feature_request,Предложение по улучшению приложения,1,очень неудобный приложение проблема внутри куч...
...,...,...,...,...,...,...,...,...
475,VK,самое худшее приложение.,1,1,rating,Краткая негативная оценка без деталей,2,самый плохой приложение
476,WhatsApp,Приложение вылетает при запуске. Mi6,1,7,bug_report,Приложение вылетает,0,приложение вылетать при запуск mi
477,VK,очень нравится приложение,5,0,rating,Краткая положительная оценка без деталей,2,очень нравиться приложение
478,Telegram,"не грузят гс, видео, фото, кружки ни с впн, ни...",4,0,user_experience,Проблемы с загрузкой медиа,3,не грузить гс видео фото кружка ни с впн ни с ...


In [14]:
train_texts = [ex for ex in train['tokens']]

counter = Counter()
for text in train_texts:
    counter.update(tokenize(text))

In [15]:
counter.most_common(20)

[(Substring(0, 2, 'не'), 160),
 (Substring(0, 5, 'очень'), 118),
 (Substring(0, 3, 'всё'), 112),
 (Substring(0, 10, 'приложение'), 91),
 (Substring(0, 7, 'хороший'), 72),
 (Substring(0, 1, 'я'), 64),
 (Substring(0, 8, 'отличный'), 54),
 (Substring(0, 7, 'спасибо'), 53),
 (Substring(0, 6, 'почему'), 53),
 (Substring(8, 18, 'приложение'), 47),
 (Substring(7, 9, 'не'), 41),
 (Substring(3, 11, 'работать'), 40),
 (Substring(0, 5, 'после'), 38),
 (Substring(9, 19, 'приложение'), 37),
 (Substring(0, 12, 'здравствуйте'), 35),
 (Substring(0, 1, 'в'), 33),
 (Substring(6, 13, 'хороший'), 30),
 (Substring(3, 7, 'мочь'), 29),
 (Substring(15, 25, 'приложение'), 28),
 (Substring(0, 14, 'отвратительный'), 28)]

## Новый вариант препроцессинга

In [16]:
train_part, valid_part = train_test_split(
    train,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=train["label_num"]
)

In [17]:
# -------------------------
# 3) TF-IDF на уже готовых токенах
#    Важно: tokenizer=str.split, потому что текст уже предтокенизирован
# -------------------------
tfidf = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None
)
tfidf.fit(train_part["tokens"])

# словарь IDF: token -> idf
idf_dict = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))


# -------------------------
# 4) Загрузка RusVectores / compressed fastText
# -------------------------
# Подставьте путь к скачанной 202MB модели:
# geowac_tokens_sg_300_5_2020-400K-100K-300.bin
wv_model = compress_fasttext.models.CompressedFastTextKeyedVectors.load(
    "geowac_tokens_sg_300_5_2020-400K-100K-300.bin"
)


# -------------------------
# 5) Векторизатор текста
#    Делает TF-IDF-взвешенное среднее word vectors
# -------------------------
def get_vectorizer(wv_model, idf_dict=None):
    dim = wv_model.vector_size

    def vectorizer(sent: str):
        tokens = [w for w in sent.split() if w in wv_model]

        if not tokens:
            return np.zeros(dim, dtype=np.float32)

        vecs = np.vstack([wv_model[w] for w in tokens]).astype(np.float32)

        if idf_dict is None:
            weights = np.ones(len(tokens), dtype=np.float32)
        else:
            weights = np.array([idf_dict.get(w, 1.0) for w in tokens], dtype=np.float32)

        # взвешенное среднее
        return np.average(vecs, axis=0, weights=weights)

    return vectorizer


vec_fn = get_vectorizer(wv_model, idf_dict)

train["ru_tfidf_vec"] = train["tokens"].apply(vec_fn)
test["ru_tfidf_vec"] = test["tokens"].apply(vec_fn)

In [18]:
train

,app_name,text,rating,thumbs_up_count,label_gold,summary_gold,label_num,tokens,ru_tfidf_vec
0,Avito,программа Авито очень помогает в продаже б.у т...,5,0,user_experience,Положительный опыт использования приложения,3,программа авить очень помогать в продажа б у т...,"[-0.021219138, 0.07674588, 0.15830201, 0.05761..."
1,VK,Скачала и пользовалась с 2015 года. Новый отзы...,1,24,user_experience,Недовольство модерацией и поддержкой,3,скачать и пользоваться с год новый отзыв писат...,"[-0.0819898, 0.078417845, 0.06648695, -1.80722..."
2,Yandex Maps,Сотрудники Яндекс Поддержки продались и сотруд...,1,5,user_experience,Мешает навязчивая реклама,3,сотрудник яндекс поддержка продаться и сотрудн...,"[-0.10758206, 0.08170811, 0.25345188, 0.070188..."
3,Telegram,"прошу, сделайте так чтоб подарки можно было пр...",5,0,feature_request,Разрешить продажу подарков в любое время,1,просить сделать так чтоб подарок можно быть пр...,"[-0.059473947, -0.073754564, 0.029559486, 0.03..."
4,Yandex Maps,настройте приложение опять заедает и вылетает,1,3,bug_report,Приложение вылетает,0,настроить приложение опять заедать и вылетать,"[-0.07277887, -0.038229764, 0.13423872, 0.1123..."
...,...,...,...,...,...,...,...,...,...
2709,Avito,сделка состоялась без проблем👍,5,0,user_experience,Положительный опыт использования приложения,3,сделка состояться без проблем👍,"[-0.11614409, 0.1174129, 0.22561292, 0.0965295..."
2710,Avito,"Заставили меня обновить приложение,хоть и не х...",1,1,bug_report,Приложение не открывается или не запускается,0,заставить я обновить приложение хоть и не хоте...,"[-0.014798784, 0.0722723, 0.13770783, -0.03366..."
2711,VK,"максимально печальное приложение, хуже просто ...",1,5,bug_report,Недовольство модерацией и поддержкой,0,максимально печальный приложение плохой просто...,"[-0.017967703, 0.058292676, 0.108313195, 0.011..."
2712,Yandex Go,"нету другого сервиса , таксисты не приезжают н...",1,0,user_experience,Недовольство платными функциями,3,нету другой сервис таксист не приезжать на лок...,"[-0.04496824, 0.04470105, 0.18276265, 0.056507..."


In [19]:
from sklearn.model_selection import train_test_split

# Разделим на Train и Test
train_index, valid_index = train_test_split(
    train.index,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=train["label_num"]
)

vectors = 'ru_tfidf_vec'
target_name = 'label_num'

# подготовим
X_train = np.vstack(train.loc[train_index, vectors].values)
X_valid = np.vstack(train.loc[valid_index, vectors].values)
y_train = train.loc[train_index, target_name].values
y_valid = train.loc[valid_index, target_name].values

# Init Model

In [20]:
# Init logreg model
logreg = LogisticRegression(
    C=2,
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_SEED
)

# Train Model

In [21]:
# Train logreg
logreg.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,2
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [22]:
from sklearn.metrics import classification_report

print(f'train acc: {logreg.score(X_train, y_train):.2f}')
print(f'valid acc: {logreg.score(X_valid, y_valid):.2f}')

train acc: 0.75
valid acc: 0.64


In [23]:
y_train_pred = logreg.predict(X_train)
y_valid_pred = logreg.predict(X_valid)

print(classification_report(y_train, y_train_pred))
print(classification_report(y_valid, y_valid_pred))

              precision    recall  f1-score   support

           0       0.71      0.81      0.76       529
           1       0.40      0.92      0.55       109
           2       0.84      0.89      0.86       431
           3       0.86      0.64      0.73       966

    accuracy                           0.75      2035
   macro avg       0.70      0.81      0.73      2035
weighted avg       0.79      0.75      0.76      2035

              precision    recall  f1-score   support

           0       0.61      0.69      0.65       176
           1       0.22      0.67      0.33        36
           2       0.77      0.73      0.75       144
           3       0.78      0.57      0.66       323

    accuracy                           0.64       679
   macro avg       0.60      0.66      0.60       679
weighted avg       0.70      0.64      0.66       679



In [29]:
label_dict

{'bug_report': 0, 'feature_request': 1, 'rating': 2, 'user_experience': 3}

## Пробуем добавить рейтинг и лайки в качестве признаков

In [28]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np

vectors = "ru_tfidf_vec"
target_name = "label_num"

train_index, valid_index = train_test_split(
    train.index,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=train[target_name]
)

X_text_train = np.vstack(train.loc[train_index, vectors].values)
X_text_valid = np.vstack(train.loc[valid_index, vectors].values)

y_train = train.loc[train_index, target_name].values
y_valid = train.loc[valid_index, target_name].values

# rating лучше закодировать one-hot, потому что 1-5 — это не обязательно линейная шкала для модели
rating_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_rating_train = rating_encoder.fit_transform(train.loc[train_index, ["rating"]])
X_rating_valid = rating_encoder.transform(train.loc[valid_index, ["rating"]])

# thumbs_up_count логарифмируем и масштабируем
thumbs_scaler = StandardScaler()

X_thumbs_train = thumbs_scaler.fit_transform(
    np.log1p(train.loc[train_index, ["thumbs_up_count"]].astype(float))
)

X_thumbs_valid = thumbs_scaler.transform(
    np.log1p(train.loc[valid_index, ["thumbs_up_count"]].astype(float))
)

# объединяем текстовые и табличные признаки
X_train = np.hstack([X_text_train, X_rating_train, X_thumbs_train])
X_valid = np.hstack([X_text_valid, X_rating_valid, X_thumbs_valid])

In [32]:
logreg_rate_thumbs = LogisticRegression(
    C=2,
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_SEED
)

In [33]:
# Train logreg
logreg_rate_thumbs.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,2
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [35]:
from sklearn.metrics import classification_report

print(f'train acc: {logreg_rate_thumbs.score(X_train, y_train):.2f}')
print(f'valid acc: {logreg_rate_thumbs.score(X_valid, y_valid):.2f}')

train acc: 0.75
valid acc: 0.65


In [36]:
y_train_pred = logreg_rate_thumbs.predict(X_train)
y_valid_pred = logreg_rate_thumbs.predict(X_valid)

print(classification_report(y_train, y_train_pred))
print(classification_report(y_valid, y_valid_pred))

              precision    recall  f1-score   support

           0       0.72      0.82      0.76       529
           1       0.41      0.92      0.56       109
           2       0.83      0.89      0.86       431
           3       0.86      0.64      0.73       966

    accuracy                           0.75      2035
   macro avg       0.70      0.82      0.73      2035
weighted avg       0.79      0.75      0.76      2035

              precision    recall  f1-score   support

           0       0.61      0.70      0.65       176
           1       0.24      0.69      0.35        36
           2       0.78      0.73      0.75       144
           3       0.79      0.58      0.67       323

    accuracy                           0.65       679
   macro avg       0.60      0.68      0.61       679
weighted avg       0.71      0.65      0.67       679



# Смотрим результаты на тестовой выборке

## Сначала на дефолтном LogReg

In [38]:
X_test = np.vstack(test.loc[:, vectors].values)
y_test = test.loc[:, target_name].values


In [42]:
y_default_pred = logreg.predict(X_test)

In [44]:
print(classification_report(y_test, y_default_pred))

              precision    recall  f1-score   support

           0       0.61      0.78      0.68       125
           1       0.28      0.69      0.40        26
           2       0.79      0.72      0.76       101
           3       0.77      0.56      0.65       228

    accuracy                           0.66       480
   macro avg       0.61      0.69      0.62       480
weighted avg       0.71      0.66      0.67       480



## Затем по LogReg с добавленными фичами

In [47]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np

vectors = "ru_tfidf_vec"
target_name = "label_num"


X_text_test = np.vstack(test.loc[:, vectors].values)
y_test = test.loc[:, target_name].values

# rating лучше закодировать one-hot, потому что 1-5 — это не обязательно линейная шкала для модели
rating_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_rating_test = rating_encoder.fit_transform(test.loc[:, ["rating"]])

# thumbs_up_count логарифмируем и масштабируем
thumbs_scaler = StandardScaler()

X_thumbs_test = thumbs_scaler.fit_transform(
    np.log1p(test.loc[:, ["thumbs_up_count"]].astype(float))
)

# объединяем текстовые и табличные признаки
X_test_featured = np.hstack([X_text_test, X_rating_test, X_thumbs_test])

In [48]:
y_default_featured = logreg_rate_thumbs.predict(X_test_featured)

In [50]:
print(classification_report(y_test, y_default_pred))

              precision    recall  f1-score   support

           0       0.61      0.78      0.68       125
           1       0.28      0.69      0.40        26
           2       0.79      0.72      0.76       101
           3       0.77      0.56      0.65       228

    accuracy                           0.66       480
   macro avg       0.61      0.69      0.62       480
weighted avg       0.71      0.66      0.67       480



In [49]:
print(classification_report(y_test, y_default_featured))

              precision    recall  f1-score   support

           0       0.58      0.79      0.67       125
           1       0.30      0.69      0.42        26
           2       0.81      0.76      0.79       101
           3       0.79      0.53      0.64       228

    accuracy                           0.66       480
   macro avg       0.62      0.69      0.63       480
weighted avg       0.71      0.66      0.66       480

